# JEPA (P2P) — Time Series Self-Supervised Pretraining + Forecasting

Simplified JEPA: patch-to-patch prediction only — no VQ codebook, no semantic tokens.

**Pipeline:** Pretrain on Monash → Zeroshot forecasting on ETTh1

## 1 — Mount Drive & set project root

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

# ── Change this to wherever you uploaded the allthree folder ──────────────────
PROJECT_ROOT = '/content/drive/MyDrive/allthree'

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

## 2 — Choose dataset

In [ ]:
# Pretrain on Monash (all .tsf files) — controlled by pretrain_on_monash in config_jepa.py
# Set FORECAST_DATASET for the downstream task.
FORECAST_DATASET = 'etth1'   # ETTh1 for forecasting evaluation

from dataset_registry import get_dataset_info
import pandas as pd

ds = get_dataset_info(FORECAST_DATASET)
df = pd.read_csv(ds['csv_path'], nrows=0)
print(f'Forecast → {FORECAST_DATASET}  {ds["c_in"]} vars   {ds["csv_path"]}')

## 3 — Check GPU

In [ ]:
import torch, sklearn
print(f'torch   {torch.__version__}')
print(f'sklearn {sklearn.__version__}')
print(f'GPU     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT available — go to Runtime > Change runtime type > T4 GPU"}')

## 4 — Verify data

In [ ]:
ds = get_dataset_info(FORECAST_DATASET)
df = pd.read_csv(ds['csv_path'])
print(f'── Forecast: {FORECAST_DATASET} ──')
print(f'  Rows    : {len(df)}')
print(f'  Columns : {df.columns.tolist()}')
print(df.head(2))

## 5 — (Optional) Preview config

Edit `JEPA/config_files/config_jepa.py` on Drive, or preview key values here:

In [ ]:
import importlib.util
from pathlib import Path

_spec = importlib.util.spec_from_file_location(
    'config_jepa', Path(PROJECT_ROOT) / 'JEPA' / 'config_files' / 'config_jepa.py')
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
cfg = _mod.config

print('JEPA config snapshot:')
for k in ['num_epochs', 'encoder_embed_dim', 'num_encoder_layers', 'nhead',
          'patch_size', 'ratio_patches', 'batch_size',
          'pretrain_on_monash', 'forecasting_modes', 'vigreg_var', 'vigreg_cov']:
    print(f'  {k:30s} = {cfg.get(k, "(not set")}')

## 6 — Run JEPA (pretrain → forecasting)

In [ ]:
from Train_and_downstream import run

run(
    model            = 'jepa_simple',
    forecast_dataset = FORECAST_DATASET,
    skip_train       = False,
)

## 7 — Forecasting only (load existing checkpoint)

Set `skip_train=True` to skip pretraining and go straight to forecasting using a saved checkpoint.

In [ ]:
# run(model='jepa_simple', forecast_dataset=FORECAST_DATASET, skip_train=True)